In [1]:
#%%configure
#{"vCores": 16}

In [2]:
!pip install duckrun --upgrade


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\mdjouallah\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import duckrun
from   psutil import *
core              = cpu_count()
Nbr_threads = core * 2
print(core)

8


### <u>_**<mark>Parameters</mark>**_</u>

In [4]:
ws                    = 'duckrun'
lh                    = 'data'
schema                = 'aemo'
business_logic        = '/fabric_demo/transformation'
nbr_days_download     =  2

# Update Data

In [5]:
con = duckrun.connect(f"{ws}/{lh}.lakehouse/{schema}", business_logic)

🔌 Attaching tables from schema: aemo
🔐 Starting Azure authentication...
🖥️ Trying local authentication (Azure CLI + browser fallback)...
🔐 Trying Azure CLI authentication...
✅ Azure CLI authentication successful!
🔍 Discovering tables via OneLake Delta Table API...
   Using identifier: duckrun/data.Lakehouse
   Listing tables in schema: aemo
   Found 8 tables
mstdatetime, scada, scada_today, summary, duid, calendar, price_today, price


In [6]:
remaining_files       =  max(0,nbr_days_download - 60)

In [7]:
nightly =[
              
              ('scraping', (["https://github.com/djouallah/fabric_demo/tree/main/data/archive/*"],
                            ["Reports/Current/Daily_Reports/"],
                            nbr_days_download,ws,lh,Nbr_threads)),
              ('price','append'),
              ('scada','append'),
              ('download_excel',("raw/", ws,lh)),
              ('duid','overwrite'),
              ('calendar','ignore'),
              ('mstdatetime','ignore'),
              ('summary__backfill','overwrite')
         ]

intraday = [
              ('scraping', (["http://nemweb.com.au/Reports/Current/DispatchIS_Reports/",
                             "http://nemweb.com.au/Reports/Current/Dispatch_SCADA/" ],
                            ["Reports/Current/DispatchIS_Reports/","Reports/Current/Dispatch_SCADA/"],
                             288, ws,lh,Nbr_threads)),
              ('price_today','append'),
              ('scada_today','append'),
              ('duid','ignore'),
              ('summary__incremental', 'append')            
          ]

history_download = [('scraping',(["https://github.com/djouallah/fabric_demo/tree/main/data/archive/*"],
                                   ["Reports/Current/Daily_Reports/"],
                                   remaining_files,ws,lh,Nbr_threads))]

history_process = [('scada','append'),('price','append'),('summary__backfill_archive','append')]

In [8]:
#create lakehouse if not exists
con.create_lakehouse_if_not_exists(lh)

🔐 Getting Fabric API token...
🖥️ Using CLI + browser fallback for Fabric API
🔐 Trying Azure CLI for Fabric API...
✅ Fabric API token obtained via Azure CLI!
Lakehouse 'data' already exists


True

In [9]:
%%time
con.run(nightly)


Task 1/8: scraping
Running Python: scraping(['https://github.com/djouallah/fabric_demo/tree/main/data/archive/*'], ['Reports/Current/Daily_Reports/'], 2, 'duckrun', 'data', 16)
Flushed 2 files to disk and updated log Reports/Current/Daily_Reports/download_log.csv
https://github.com/djouallah/fabric_demo/tree/main/data/archive/* - 2 files extracted and uploaded
✅ Python 'scraping' completed

Task 2/8: price


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ SQL 'price' → 'price' (append) (engine=pyarrow)

Task 3/8: scada


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ SQL 'duid' → 'duid' (overwrite) (engine=pyarrow)

Task 6/8: calendar
Table calendar exists. Skipping (mode='ignore')
✅ SQL 'calendar' → 'calendar' (ignore) (engine=pyarrow)

Task 7/8: mstdatetime
Table mstdatetime exists. Skipping (mode='ignore')
✅ SQL 'mstdatetime' → 'mstdatetime' (ignore) (engine=pyarrow)

Task 8/8: summary__backfill


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ SQL 'summary__backfill' → 'summary' (overwrite) (engine=pyarrow)

✅ All tasks completed successfully
CPU times: total: 45.2 s
Wall time: 10min 59s


True

In [10]:
%%time
con.run(intraday)


Task 1/5: scraping
Running Python: scraping(['http://nemweb.com.au/Reports/Current/DispatchIS_Reports/', 'http://nemweb.com.au/Reports/Current/Dispatch_SCADA/'], ['Reports/Current/DispatchIS_Reports/', 'Reports/Current/Dispatch_SCADA/'], 288, 'duckrun', 'data', 16)
Failed to download PUBLIC_DISPATCHIS_202601131500_0000000498559287.zip: HTTP 403
Failed to download PUBLIC_DISPATCHIS_202601131410_0000000498553882.zip: HTTP 403
Failed to download PUBLIC_DISPATCHIS_202601131400_0000000498552724.zip: HTTP 403
Failed to download PUBLIC_DISPATCHIS_202601131350_0000000498551653.zip: HTTP 403
Failed to download PUBLIC_DISPATCHIS_202601131345_0000000498551088.zip: HTTP 403
Failed to download PUBLIC_DISPATCHIS_202601131340_0000000498550594.zip: HTTP 403
Failed to download PUBLIC_DISPATCHIS_202601131335_0000000498549924.zip: HTTP 403
Failed to download PUBLIC_DISPATCHSCADA_202601131800_0000000498579361.zip: HTTP 403
Failed to download PUBLIC_DISPATCHSCADA_202601131805_0000000498579908.zip: HTTP 40

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ SQL 'price_today' → 'price_today' (append) (engine=pyarrow)

Task 3/5: scada_today


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ SQL 'scada_today' → 'scada_today' (append) (engine=pyarrow)

Task 4/5: duid
Table duid exists. Skipping (mode='ignore')
✅ SQL 'duid' → 'duid' (ignore) (engine=pyarrow)

Task 5/5: summary__incremental


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ SQL 'summary__incremental' → 'summary' (append) (engine=pyarrow)

✅ All tasks completed successfully
CPU times: total: 13.7 s
Wall time: 2min 35s


True

In [11]:
%%time
if remaining_files > 0:
    con.run(history_download)

CPU times: total: 0 ns
Wall time: 0 ns


In [12]:
%%time
if remaining_files > 0:
    con.run(history_process)

CPU times: total: 0 ns
Wall time: 0 ns


In [13]:
totalrows=(con.sql("select count(*) from summary").fetchone()[0])
print(totalrows)

1303775
